# 🧪 Baseline: Plain Qwen2.5-3B BEFORE Fine-Tuning

This notebook loads the **raw, unmodified** `Qwen2.5-3B-Instruct` model (no LoRA adapters, no CUAD training)
and runs it on the **exact same 3 test clauses** used in the fine-tuning evaluation.

**Purpose:** Capture the baseline "before" responses to compare side-by-side with our fine-tuned model in the README.

---
### ⚠️ Instructions
1. Open this notebook in **Google Colab** with a **T4 GPU** runtime (`Runtime > Change runtime type > T4 GPU`)
2. Run all cells in order
3. **Copy the complete output of CELL 4** and paste it into `output/before_finetune_result.txt`

---
## CELL 1: Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

---
## CELL 2: Check GPU

In [ ]:
import torch

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("ERROR: No GPU found! Go to Runtime > Change runtime type > T4 GPU")

---
## CELL 3: Load the BASE Qwen2.5-3B-Instruct (NO fine-tuning, NO LoRA)

Loads `unsloth/Qwen2.5-3B-Instruct` — the raw base model with zero legal fine-tuning.

In [ ]:
from unsloth import FastLanguageModel

print("Loading plain Qwen2.5-3B-Instruct (base model, NO fine-tuning)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print("\n✅ Base model loaded successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\n📌 This is the RAW base model — zero legal domain training.")

---
## CELL 4: Run the SAME 3 Test Clauses (COPY THIS OUTPUT)
Uses identical prompt, test clauses, and token limits.

In [ ]:
import json, time

# ── Exact same system prompt used in training (02_format_dataset.py) ────────
SYSTEM_PROMPT = """You are a legal contract clause analyzer. When given a contract clause, you must:

1. Identify the clause type from these categories: Document Name, Parties, Agreement Date, Effective Date, Expiration Date, Renewal Term, Notice Period To Terminate Renewal, Governing Law, Most Favored Nation, Non-Compete, Exclusivity, No-Solicit Of Customers, Competitive Restriction Exception, No-Solicit Of Employees, Non-Disparagement, Termination For Convenience, Rofr/Rofo/Rofn, Change Of Control, Anti-Assignment, Revenue/Profit Sharing, Price Restrictions, Minimum Commitment, Volume Restriction, Ip Ownership Assignment, Joint Ip Ownership, License Grant, Non-Transferable License, Affiliate License-Licensor, Affiliate License-Licensee, Unlimited/All-You-Can-Eat-License, Irrevocable Or Perpetual License, Source Code Escrow, Post-Termination Services, Audit Rights, Uncapped Liability, Cap On Liability, Liquidated Damages, Warranty Duration, Insurance, Covenant Not To Sue, Third Party Beneficiary.

2. Extract the key clause text and important terms.

3. Assess the risk level (HIGH, MEDIUM, or LOW).

4. Provide a plain English explanation that a non-lawyer can understand.

Respond in JSON format."""

# ── Exact same test clauses ────────────────────────────────────────────────
TEST_CLAUSES = [
    {
        "id": 1,
        "name": "Termination for Convenience Clause",
        "expected_type": "Termination For Convenience",
        "expected_risk": "HIGH",
        "text": (
            "Either party may terminate this Agreement for any reason or no reason whatsoever \n"
            "    upon thirty (30) days' prior written notice to the other party. Upon such termination, \n"
            "    all licenses granted hereunder shall immediately cease, and each party shall promptly \n"
            "    return or destroy all Confidential Information of the other party."
        ),
    },
    {
        "id": 2,
        "name": "Governing Law Clause",
        "expected_type": "Governing Law",
        "expected_risk": "LOW",
        "text": (
            "This Agreement shall be governed by and construed in accordance with \n"
            "    the laws of the State of Delaware, without regard to its conflict of \n"
            "    laws principles."
        ),
    },
    {
        "id": 3,
        "name": "Cap on Liability Clause",
        "expected_type": "Cap On Liability",
        "expected_risk": "HIGH",
        "text": (
            "IN NO EVENT SHALL EITHER PARTY'S TOTAL LIABILITY UNDER THIS AGREEMENT \n"
            "    EXCEED THE TOTAL FEES PAID BY CUSTOMER DURING THE TWELVE (12) MONTH \n"
            "    PERIOD IMMEDIATELY PRECEDING THE EVENT GIVING RISE TO SUCH LIABILITY."
        ),
    },
]

def run_base_model(clause_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze the following contract clause and identify any relevant legal provisions:\n\n{clause_text}"},
    ]
    
    # return_dict=True provides both input_ids and attention_mask to prevent warnings
    inputs = tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True, 
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")
    
    t0 = time.time()
    outputs = model.generate(
        **inputs, 
        max_new_tokens=768,       # Generous limit: prevents truncations
        temperature=0.1, 
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return response, elapsed


print("=" * 70)
print("  BASELINE: Plain Qwen2.5-3B-Instruct (BEFORE Fine-Tuning)")
print("  Model : unsloth/Qwen2.5-3B-Instruct  |  No CUAD training")
print("=" * 70)

baseline_results = []

for test in TEST_CLAUSES:
    print(f"\n{'=' * 60}")
    print(f"  TEST {test['id']}: {test['name']}")
    print(f"  Expected Clause Type : {test['expected_type']}")
    print(f"  Expected Risk Level  : {test['expected_risk']}")
    print(f"{'=' * 60}")

    raw, elapsed = run_base_model(test["text"])

    parsed = None
    try:
        parsed = json.loads(raw)
    except Exception:
        if "```json" in raw:
            try:
                parsed = json.loads(raw.split("```json")[1].split("```")[0].strip())
            except Exception:
                pass
        elif "{" in raw and "}" in raw:
            try:
                parsed = json.loads(raw[raw.index("{"):raw.rindex("}")+1])
            except Exception:
                pass
    if parsed is None:
        parsed = {"raw_response": raw}

    baseline_results.append(parsed)

    print(f"\n  ── BASE MODEL RESPONSE ({elapsed:.1f}s) ──")
    if "raw_response" not in parsed:
        print(json.dumps(parsed, indent=2))
    else:
        print(raw)

    got_type = parsed.get("clause_type", "") if isinstance(parsed, dict) else ""
    got_risk = parsed.get("risk_level", "") if isinstance(parsed, dict) else ""
    
    type_ok = (
        test["expected_type"].lower() in got_type.lower() 
        or got_type.lower() in test["expected_type"].lower()
    )
    risk_ok = got_risk.upper() == test["expected_risk"].upper()
    valid_json = isinstance(parsed, dict) and "raw_response" not in parsed

    print(f"\n  ── Quick Check ──")
    print(f"  Clause Type  → got '{got_type}' | expected '{test['expected_type']}' → {'✅' if type_ok else '❌'}")
    print(f"  Risk Level   → got '{got_risk}' | expected '{test['expected_risk']}' → {'✅' if risk_ok else '❌'}")
    print(f"  Valid JSON   → {'✅' if valid_json else '❌ (not valid JSON)'}")

print(f"\n{'=' * 70}")
print("  Done! Copy the output above into output/before_finetune_result.txt")
print(f"{'=' * 70}")

---
## CELL 5: Save Results to JSON (optional)

In [ ]:
import json

output = {
    "model": "unsloth/Qwen2.5-3B-Instruct (base, no fine-tuning)",
    "note": "Baseline responses BEFORE fine-tuning on CUAD dataset",
    "baseline_results": baseline_results,
}

with open("/content/baseline_results.json", "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Saved to /content/baseline_results.json")